# Notebook 15: Cross-Modal Retrieval

## Why This Matters
CLIP maps images and text to the same embedding space. This enables retrieval across modalities: find images matching a text description, find captions matching an image, or rank retrieved documents by visual-textual relevance. Cross-modal retrieval is the foundation of image search engines, multimodal RAG, and production content moderation systems.

## Structure
1. Cross-modal embedding alignment (narrative)
2. Text → Image retrieval pipeline
3. Image → Text retrieval
4. Evaluating cross-modal recall@k
5. Asymmetric retrieval — query and corpus use different encoders
6. Bridge to model fingerprinting

## What You'll Learn
- How CLIP's shared embedding space enables cross-modal similarity
- How to evaluate cross-modal retrieval with recall@k
- When cross-modal retrieval fails and how to diagnose it


## Background

### Cross-modal alignment

CLIP's training objective explicitly aligns image and text representations: matched (image, caption) pairs have high cosine similarity, unmatched pairs have low similarity. The result is a shared semantic space where:

```
cosine_similarity(encode_image("dog_photo.jpg"), encode_text("a photo of a dog")) ≈ 0.9
cosine_similarity(encode_image("dog_photo.jpg"), encode_text("a photo of a cat")) ≈ 0.5
```

This enables cross-modal retrieval: encode a text query and find the nearest image embeddings (or vice versa). The alignment isn't perfect — CLIP was trained on internet data with noisy captions — but it generalizes remarkably well.

### Asymmetric retrieval

In many production systems, queries and corpus items are different types:
- User queries are short text; corpus items are images
- User uploads an image; corpus items are text descriptions

These can use different encoders (query encoder vs document encoder) that are jointly trained. Dense Passage Retrieval (DPR) used this for text QA; in cross-modal retrieval CLIP handles both but the two encoders (image and text) naturally play these asymmetric roles.


In [ ]:
# %pip install -q openai-clip torch torchvision numpy matplotlib scikit-learn Pillow requests

In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import clip
from torch.utils.data import DataLoader

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print(f"CLIP ViT-B/32 loaded on {device}")

CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

## 1. Building a Cross-Modal Index

In [ ]:
# Encode a gallery of images once — this is done offline in production
cifar_test = torchvision.datasets.CIFAR10("/tmp/cifar10", train=False, download=True,
                                           transform=preprocess)
gallery_loader = DataLoader(cifar_test, batch_size=256, shuffle=False, num_workers=0)

print("Encoding gallery images...")
img_features_list, img_labels = [], []
with torch.no_grad():
    for imgs, labels in gallery_loader:
        feats = clip_model.encode_image(imgs.to(device))
        feats = feats / feats.norm(dim=-1, keepdim=True)
        img_features_list.append(feats.cpu()); img_labels.append(labels)

img_features = torch.cat(img_features_list)  # [N, 512]
img_labels   = torch.cat(img_labels).numpy()
print(f"Gallery: {img_features.shape[0]:,} images × {img_features.shape[1]} dims")

## 2. Text → Image Retrieval

In [ ]:
def text_to_image(query: str, img_features, img_labels, top_k=5):
    tokens = clip.tokenize([query]).to(device)
    with torch.no_grad():
        q_feat = clip_model.encode_text(tokens)
        q_feat = q_feat / q_feat.norm(dim=-1, keepdim=True)
    sims = (q_feat.cpu() @ img_features.T).squeeze(0).numpy()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(idx, sims[idx], CLASSES[img_labels[idx]]) for idx in top_idx]


queries_with_expected = [
    ("a furry four-legged animal",             {"dog","cat","deer","horse"}),
    ("a vehicle that flies through the air",   {"airplane"}),
    ("something that lives in the ocean",      {"ship"}),
    ("a small animal that hops and croaks",    {"frog"}),
    ("a large powerful animal with a mane",    {"horse","deer"}),
]

print("Text → Image retrieval:")
print(f"{'Query':<40} {'Top-3 retrieved classes':<30} {'Recall@1'}")
print("-" * 80)
for query, expected in queries_with_expected:
    results = text_to_image(query, img_features, img_labels, top_k=5)
    top3 = [cls for _, _, cls in results[:3]]
    r1   = "✓" if results[0][2] in expected else "✗"
    print(f"  {query[:38]:<40} {str(top3):<30} {r1}")

## 3. Image → Text Retrieval

In [ ]:
# Encode a text corpus
text_corpus = [
    f"a photo of a {cls}" for cls in CLASSES
] + [
    "a small furry mammal", "a large animal with four legs",
    "something that flies", "a water vehicle",
    "transportation on land", "a wild forest animal",
]

text_tokens = clip.tokenize(text_corpus).to(device)
with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

def image_to_text(img_idx: int, text_features, text_corpus, top_k=3):
    img_feat = img_features[img_idx:img_idx+1]
    sims = (img_feat @ text_features.cpu().T).squeeze(0).numpy()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(text_corpus[i], sims[i]) for i in top_idx]


print("Image → Text retrieval (sample images):")
print()
for idx in [0, 100, 200, 500, 1000]:
    true_class = CLASSES[img_labels[idx]]
    results = image_to_text(idx, text_features.cpu(), text_corpus)
    top3 = [f"{t[:30]!r}({s:.2f})" for t, s in results]
    print(f"  Image [{idx}] (true: {true_class:12s}): {top3}")

## 4. Cross-Modal Recall@k Evaluation

In [ ]:
# Formal evaluation: for each class, can text queries retrieve images of that class?
def cross_modal_recall(img_features, img_labels, clip_model, device, k=5):
    recalls = {}
    for cls_idx, cls_name in enumerate(CLASSES):
        # Text query
        query = f"a photo of a {cls_name}"
        tokens = clip.tokenize([query]).to(device)
        with torch.no_grad():
            q_feat = clip_model.encode_text(tokens)
            q_feat = q_feat / q_feat.norm(dim=-1, keepdim=True)

        sims = (q_feat.cpu() @ img_features.T).squeeze(0).numpy()
        top_k_labels = img_labels[np.argsort(sims)[::-1][:k]]
        recall = (top_k_labels == cls_idx).mean()
        recalls[cls_name] = recall

    return recalls


print(f"Cross-modal recall@5 (text → image, CIFAR-10):")
recalls = cross_modal_recall(img_features, img_labels, clip_model, device, k=5)
for cls, rec in sorted(recalls.items(), key=lambda x: -x[1]):
    bar = '█' * int(rec * 20)
    print(f"  {cls:12s}: {rec:.3f} {bar}")
print(f"\nMean recall@5: {np.mean(list(recalls.values())):.3f}")

## 5. Bridge to Model Fingerprinting

Cross-modal retrieval requires that both modalities map to the same geometry. This means the embedding geometry is a fingerprint of the model — two different CLIP models will map the same image to different points in embedding space, and the geometry of those points encodes information about which model produced them.

Notebook 16 exploits this: given only black-box access to an embedding model (or the output embeddings), can you infer which model produced them? This is the `whodunit` foundation.


## Summary

| Concept | Key Point |
|---------|-----------|
| Shared embedding space | CLIP aligns image and text into the same cosine-similar space |
| Text→image retrieval | Encode query with text encoder, search image index |
| Image→text retrieval | Encode query with image encoder, search text index |
| Cross-modal recall@k | Fraction of queries whose correct answer appears in top-k |
| Failure modes | Low-frequency classes, domain shift from CLIP's training distribution |

**Next:** Notebook 16 — Model Fingerprinting via Embeddings. Using embedding geometry to infer which model produced a set of representations — the `whodunit` foundation.
